# 美联储G.17工业生产同版数值与发布间修订核验

只读已封存Qlib Recorder中的192份NAICS3344数值事件。独立复算当版环比，再比较相邻发布在同一观测月的值，基年变换单独记录；不读SOXX结果、不拟合。

In [1]:
import torch
import json,hashlib
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.data.minute_intake import atomic_json
root=Path.cwd();card=root/'research/semiconductor_ip_g17_vintage_revision_audit_20261022';plan=json.loads((card/'plan.json').read_text());parent=plan['parent_source'];parent_card=root/parent['card']
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert sha(parent_card/'summary.json')==parent['summary_sha256']
assert sha(parent_card/'verification_manifest.json')==parent['manifest_sha256']
artifact=parent_card/'qlib_recorder'/parent['experiment_id']/parent['recorder_id']/'artifacts'/parent['event_artifact']
assert sha(artifact)==parent['event_artifact_sha256']
parent_summary=json.loads((parent_card/'summary.json').read_text());assert parent_summary['nas_evidence']['sha256']==parent['nas_archive_sha256']
qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
with R.uri_context(parent_summary['recorder_uri']):
 rec=R.get_recorder(recorder_id=parent['recorder_id'],experiment_id=parent['experiment_id'])
 events=rec.load_object(parent['event_artifact'])
assert len(events)==plan['expected_events']
events=events.sort_values('release_date').reset_index(drop=True)
assert pd.to_datetime(events.release_date).is_monotonic_increasing
latest=events.ip_latest_index.to_numpy(float);prior=events.ip_prior_index_same_vintage.to_numpy(float)
recorded=events.ip_mom_pct_same_vintage.to_numpy(float)
assert np.isfinite(latest).all() and np.isfinite(prior).all() and (latest>0).all() and (prior>0).all()
recomputed=100*(latest/prior-1)
error=np.abs(recomputed-recorded)
assert np.isfinite(recorded).all() and float(error.max())<1e-9
base_counts={str(int(k)):int(v) for k,v in events.index_base_year.value_counts().sort_index().items()}
assert base_counts=={'2007':59,'2012':70,'2017':63}
comparisons=[];unmatched=[]
for index in range(1,len(events)):
 old,new=events.iloc[index-1],events.iloc[index]
 target=pd.Timestamp(old.observation_month).to_period('M')
 months=[pd.Timestamp(value).to_period('M') for value in new.published_months]
 hits=[i for i,month in enumerate(months) if month==target]
 if len(hits)!=1:
  unmatched.append({'earlier_release':str(pd.Timestamp(old.release_date).date()),
                    'later_release':str(pd.Timestamp(new.release_date).date()),
                    'old_latest_month':str(target),'hits':len(hits)})
  continue
 old_level=float(old.ip_latest_index);new_level=float(new.published_SA_indices[hits[0]])
 assert old_level>0 and new_level>0
 comparisons.append({'earlier_release':str(pd.Timestamp(old.release_date).date()),
  'later_release':str(pd.Timestamp(new.release_date).date()),'same_observation_month':str(target),
  'earlier_level':old_level,'later_level':new_level,
  'earlier_base_year':int(old.index_base_year),'later_base_year':int(new.index_base_year),
  'same_base_year':bool(old.index_base_year==new.index_base_year),
  'later_over_earlier_level':new_level/old_level,
  'absolute_revision_pct_same_base':abs(100*(new_level/old_level-1)) if old.index_base_year==new.index_base_year else None})
same=[x for x in comparisons if x['same_base_year']]
switches=[x for x in comparisons if not x['same_base_year']]
assert len(switches)==2 and len(comparisons)+len(unmatched)==191
revision=[x['absolute_revision_pct_same_base'] for x in same]
result={'status':'same_vintage_arithmetic_verified_adjacent_release_revisions_and_base_switches_measured',
 'plan_sha256':sha(card/'plan.json'),'source_recorder_id':parent['recorder_id'],'source_events':len(events),
 'same_vintage_mom_max_abs_recompute_error_pct':float(error.max()),
 'base_year_event_counts':base_counts,'adjacent_release_pairs':191,
 'same_observation_month_comparisons':len(comparisons),'unmatched_pairs':unmatched,
 'same_base_comparisons':len(same),'same_base_nonzero_revision_count':sum(x>1e-12 for x in revision),
 'same_base_median_abs_revision_pct':float(np.median(revision)),
 'same_base_max_abs_revision_pct':float(np.max(revision)),
 'same_base_abs_revision_ge_5pct':sum(x>=5 for x in revision),
 'same_base_abs_revision_ge_10pct':sum(x>=10 for x in revision),
 'same_base_top10_absolute_revisions':sorted(same,key=lambda row:row['absolute_revision_pct_same_base'],reverse=True)[:10],
 'base_switches':switches,
 'historical_first_receipt_qualified':False,'definition_continuity_qualified':False,
 'market_outcome_reads':0,'new_model_fits':0,'new_native_accounts':0,
 'limitation':'Within-release percentage change removes the index unit but not possible methodology or industry-composition revisions; adjacent-vintage comparisons are descriptive source checks, not predictive evidence.'}
atomic_json(result,card/'revision_audit.json')
result


{'status': 'same_vintage_arithmetic_verified_adjacent_release_revisions_and_base_switches_measured',
 'plan_sha256': 'ba50778b573558fe942c88cf7543cbb526745ca8f0549dab66c9afdce249b144',
 'source_recorder_id': '06714d1db51e487aa095d918a7eaca69',
 'source_events': 192,
 'same_vintage_mom_max_abs_recompute_error_pct': 0.0,
 'base_year_event_counts': {'2007': 59, '2012': 70, '2017': 63},
 'adjacent_release_pairs': 191,
 'same_observation_month_comparisons': 191,
 'unmatched_pairs': [],
 'same_base_comparisons': 189,
 'same_base_nonzero_revision_count': 188,
 'same_base_median_abs_revision_pct': 1.3504388926401045,
 'same_base_max_abs_revision_pct': 63.13945339873863,
 'same_base_abs_revision_ge_5pct': 11,
 'same_base_abs_revision_ge_10pct': 6,
 'same_base_top10_absolute_revisions': [{'earlier_release': '2013-03-15',
   'later_release': '2013-04-16',
   'same_observation_month': '2013-02',
   'earlier_level': 142.7,
   'later_level': 232.8,
   'earlier_base_year': 2007,
   'later_base_year':

## 中文结论

只读原生Qlib来源Recorder的192个NAICS3344工业生产发布事件；当版“最新月/上月−1”环比独立复算最大绝对误差为0。191组相邻发布都能在后一版找到前版最新观测月。

同基年189组中188组该观测月数值发生非零修订；绝对修订中位数 1.350%，11组至少5%，6组至少10%。最大的一组从2013-03-15的142.7改到2013-04-16的232.8，达 63.14%。官方[2013年度修订](https://www.federalreserve.gov/Releases/g17/articles/rev2013/industrial13.htm)与此时段相邻，但这里不把幅度全部归因于单一方法变化。

两次指数基年切换：2015-07至08从2007基变2012基，同一2015-06月水平比0.3408；2021-05至06从2012基变2017基，同一2021-04月水平比0.6267。跨版绝对指数显然不可直接拼接。当版环比虽消除了同版计量基数，却不能自动消除行业组成、季调或方法修订。

决定 `source_revision_evidence_requires_same_vintage_only_no_fit`。本卡0市场结果、0拟合、0账户，不声称IP能预测SOXX或支持0bp。下一步须待Stage1原生日历可用时点对齐完成，再另行冻结匹配信息检验；只消费每次当版已发布的环比，不回填最新FRED修订，也不以年度修订时点挑收益窗口。来源原件及数值Recorder已按父卡NAS回读保存。
